In [3]:
import requests
import json
from datetime import datetime
import time
import random
import asyncio, aiohttp



In [4]:


def load_posts_dict(file_path):
    """
    Reads a .jsonl file where each line is a JSON object representing a post.
    Returns a dict mapping post_id (or URI) -> post data.

    Supports multiple Bluesky data formats:
      - post["uri"]
      - post["post"]["uri"]
      - post["id"]
    """
    posts_dict = {}
    bad_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                bad_lines += 1
                continue

            # Try to find a reliable ID/URI key
            post_id = (
                post.get("uri")
                or post.get("post", {}).get("uri")
                or post.get("id")
                or f"line_{line_num}"
            )

            posts_dict[post_id] = post

    print(f"✅ Loaded {len(posts_dict)} posts from {file_path} ({bad_lines} bad lines skipped)")
    return posts_dict


In [5]:
posts_dict = load_posts_dict("breachdallas.jsonl")

✅ Loaded 3000 posts from breachdallas.jsonl (0 bad lines skipped)


In [22]:


API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"


async def fetch_reposters(session, uri, limit=100):
    """Fetch reposters for one post URI safely."""
    params = {"uri": uri, "limit": limit}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        async with session.get(API_URL, params=params, headers=headers, timeout=100) as r:
            if r.status != 200:
                print(f"⚠️ HTTP {r.status} for {uri}")
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception as e:
        print(f"Error fetching {uri}: {e}")
        return []


async def collect_reposter_dict(posts_dict, concurrency=25):
    """
    Takes a dict of posts:
        { post_uri: {...}, post_uri2: {...}, ... }

    Returns a dict mapping each reposter DID -> [list of post URIs they reposted].
    Only includes posts that have repostCount > 0.
    """
    posts_to_check, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:

        for post_uri, post in posts_dict.items():
            # Detect repost count (support multiple schema shapes)
            repost_count = (
                post.get("repostCount")
                or post.get("metrics", {}).get("repostCount")
                or post.get("post", {}).get("repostCount")
                or 0
            )

            if repost_count > 0:
                posts_to_check.append(post_uri)
                tasks.append(fetch_reposters(session, post_uri))

        # Run all requests concurrently
        repost_lists = await asyncio.gather(*tasks, return_exceptions=True)

    # Merge results into reposter → [post_uris] mapping
    reposter_dict = {}
    for post_uri, reposters in zip(posts_to_check, repost_lists):
        if not reposters or isinstance(reposters, Exception):
            continue
        for reposter in reposters:
            reposter_dict.setdefault(reposter, []).append(post_uri)

    print(f"✅ Processed {len(posts_to_check)} posts. Found {len(reposter_dict)} unique reposters.")
    return reposter_dict


In [10]:
repost_dict = await collect_reposter_dict(posts_dict)

✅ Processed 579 posts. Found 199 unique reposters.


In [68]:
print(repost_dict)

{'did:plc:oouxpk5mwn55tcyko2flvh7d': ['at://did:plc:3jod2mt7f7jva63p5wagydbv/app.bsky.feed.post/3m3nzkb7b6c2v', 'at://did:plc:xweglghjgfbbqpw52pb3czlk/app.bsky.feed.post/3m2kyxwii3k2q', 'at://did:plc:qbg35y66bow3mncl2egk4asq/app.bsky.feed.post/3m2kxodsur224', 'at://did:plc:tbqwv2jbipnxzc25r4habi7z/app.bsky.feed.post/3m2kvaftkrk2s', 'at://did:plc:voqciwxnfgecsmgapzyncxry/app.bsky.feed.post/3lzzerhilxk2h', 'at://did:plc:b5vbgntfozdqcr7d53oclqb5/app.bsky.feed.post/3lzzdskdscs2k', 'at://did:plc:mc7lbdf7bzuvcslvyvtyj2gf/app.bsky.feed.post/3lzzcx2ck5s2u', 'at://did:plc:hcpkl7rhhyllabcf6xducid5/app.bsky.feed.post/3lzzcm4l55s2u', 'at://did:plc:d72m7oobo5kz65bje7mbdnku/app.bsky.feed.post/3lzzbzann5c2l', 'at://did:plc:b5vbgntfozdqcr7d53oclqb5/app.bsky.feed.post/3lzzbxervgk2k', 'at://did:plc:j5uxfis4uph2a23k4csfjcga/app.bsky.feed.post/3lzz6d5w3ec2i', 'at://did:plc:zsjcj2fopbu2b67yzufbw4ik/app.bsky.feed.post/3lzz6bt3a6c2x'], 'did:plc:gntc2im4bglfkgaiktcrem4y': ['at://did:plc:2f5xrk2cvjsds5vu46eik3

In [ ]:
import aiohttp
import asyncio

GET_FOLLOWS_URL = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"


async def get_follows(session, reposter_did):
    """Return set of DIDs the given user follows."""
    follows = set()
    params = {"actor": reposter_did, "limit": 100}
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        while True:
            async with session.get(GET_FOLLOWS_URL, params=params, headers=headers, timeout=30) as r:
                if r.status != 200:
                    print(f"⚠️ HTTP {r.status} getting follows for {reposter_did}")
                    return follows  # Return what we got

                data = await r.json()
                for user in data.get("follows", []):
                    did = user.get("did")
                    if did:
                        follows.add(did)

                cursor = data.get("cursor")
                if not cursor:
                    break
                params["cursor"] = cursor

        return follows

    except Exception as e:
        print(f"❌ Error fetching follows for {reposter_did}: {e}")
        return follows


async def collect_following_map(reposter_dict, concurrency=25):
    """
    Given:
      - reposter_dict: { reposter_did: [...] }

    Returns:
      - { reposter_did: set of followed user DIDs }
    """
    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    followers_map = {}

    reposter_dids = list(reposter_dict.keys())

    async with aiohttp.ClientSession(connector=connector) as session:
        sem = asyncio.Semaphore(concurrency)

        async def task_wrapper(reposter):
            async with sem:
                follows = await get_follows(session, reposter)
                followers_map[reposter] = follows

        tasks = [asyncio.create_task(task_wrapper(r)) for r in reposter_dids]
        await asyncio.gather(*tasks)

    print(f"✅ Done. Collected follow lists for {len(followers_map)} reposters.")
    return followers_map


In [92]:
import aiohttp
import asyncio

FOLLOW_API = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

def get_author_dids(posts_dict):
    """Extract all unique author DIDs from post data."""
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors

async def fetch_follows(session, reposter, author_set):
    """Fetch users the reposter follows and return intersection with author_set."""
    follows = set()
    cursor = None
    headers = {"User-Agent": "Mozilla/5.0"}

    while True:
        params = {"actor": reposter, "limit": 100}
        if cursor:
            params["cursor"] = cursor

        try:
            async with session.get(FOLLOW_API, params=params, headers=headers, timeout=15) as r:
                if r.status != 200:
                    print(f"⚠️ HTTP {r.status} for {reposter}")
                    return reposter, []

                data = await r.json()
                for user in data.get("follows", []):
                    did = user.get("did")
                    if did in author_set:
                        follows.add(did)

                cursor = data.get("cursor")
                if not cursor:
                    break
        except Exception as e:
            print(f"❌ Error for {reposter}: {e}")
            break

    return reposter, list(follows)

async def collect_reposter_followed_authors(reposter_dict, posts_dict, concurrency=15):
    reposters = list(reposter_dict.keys())
    author_set = get_author_dids(posts_dict)

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    results = {}

    async with aiohttp.ClientSession(connector=connector) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem:
                return await fetch_follows(session, reposter, author_set)

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks)

    return {r: authors for r, authors in responses if authors}


In [93]:
followers = await collect_reposter_followed_authors(repost_dict,posts_dict)

In [107]:
def find_negative_instances(reposter_did, follow_map, posts_dict, reposter_dict, num_of_instances):
    """
    Returns (post_uri, reposter_did) if there's a post whose author is followed
    by the reposter, but that post was NOT reposted by them.
    """
    followed_authors = set(follow_map.get(reposter_did, []))
    reposted_posts = set(reposter_dict.get(reposter_did, []))

    instances = []
    for post_uri, post in posts_dict.items():
        if num_of_instances == 0: break
        author = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if not author or author not in followed_authors:
            continue
        if post_uri not in reposted_posts:
            instances.append(post_uri)
            num_of_instances-=1
        
    return instances


In [109]:
for i in range(len(repost_dict)):
    reposter = list(repost_dict.keys())[i]
    neg = find_negative_instances(reposter,followers,posts_dict,repost_dict,5)
    print(neg)

['at://did:plc:scpuf7h6dsinnd23jdvr4gez/app.bsky.feed.post/3m3nm56db5s2y', 'at://did:plc:scpuf7h6dsinnd23jdvr4gez/app.bsky.feed.post/3m3nludjif22y', 'at://did:plc:b5vbgntfozdqcr7d53oclqb5/app.bsky.feed.post/3m34ekk6yqs2o', 'at://did:plc:juclc2yyslcfg7mj3sz5v4u2/app.bsky.feed.post/3m34bfyc3ws2q', 'at://did:plc:scpuf7h6dsinnd23jdvr4gez/app.bsky.feed.post/3m33icxmr7s2d']
['at://did:plc:fzux5vvnwhysex4pvnyw3jov/app.bsky.feed.post/3m3njgqe5os2d', 'at://did:plc:fzux5vvnwhysex4pvnyw3jov/app.bsky.feed.post/3m3nje4zhpc2d', 'at://did:plc:fzux5vvnwhysex4pvnyw3jov/app.bsky.feed.post/3m3ng6tho6s23', 'at://did:plc:ycl4yqijfdvihnyljhqavjay/app.bsky.feed.post/3m36vuo2itk2i', 'at://did:plc:p42p6sf72zmlx3bwukcvoiuy/app.bsky.feed.post/3m2l2sn6ndk23']
[]
['at://did:plc:ifeda2auw5iicswhcdtgelai/app.bsky.feed.post/3m2lfirt42c2b', 'at://did:plc:ifeda2auw5iicswhcdtgelai/app.bsky.feed.post/3m2ldt4eqzf2p', 'at://did:plc:ifeda2auw5iicswhcdtgelai/app.bsky.feed.post/3m2l6vfsjmg25', 'at://did:plc:ifeda2auw5iicswhcd

In [69]:
def compute_follow_ratio(follows_map):
    """Compute the overall ratio of True / (True + False) across all authors."""
    total_true = 0
    total_count = 0

    for reposters in follows_map.values():
        for follows in reposters.values():
            total_count += 1
            if follows:
                total_true += 1

    return total_true / total_count if total_count > 0 else 0.0


compute_follow_ratio(followers)

0.618798955613577